# **ACCIDENT SEVERITY PREDICTION**

# **1) Import Datasets From Kaggle**

In [1]:
import pandas as pd
import kagglehub

# Download dataset from Kaggle
dataset_path = kagglehub.dataset_download('silicon99/dft-accident-data')

# Read CSVs safely
Accidents = pd.read_csv(dataset_path + '/Accidents0515.csv',
                        index_col='Accident_Index',
                        on_bad_lines='skip',
                        low_memory=False)

Casualties = pd.read_csv(dataset_path + '/Casualties0515.csv',
                         index_col='Accident_Index',
                         on_bad_lines='skip',
                         low_memory=False)

Vehicles = pd.read_csv(dataset_path + '/Vehicles0515.csv',
                       index_col='Accident_Index',
                       on_bad_lines='skip',
                       low_memory=False)

print("✅ Datasets imported successfully!\n")

# Print shapes and previews
print(f"🚗 Accidents dataset shape: {Accidents.shape}")
display(Accidents.head(3))

print(f"\n🧍 Casualties dataset shape: {Casualties.shape}")
display(Casualties.head(3))

print(f"\n🚙 Vehicles dataset shape: {Vehicles.shape}")
display(Vehicles.head(3))


ModuleNotFoundError: No module named 'kagglehub'

# **2) Merging Datasets**

In [ ]:
# Merge Accidents with Vehicles
merged_data = pd.merge(Accidents, Vehicles,
                       left_index=True, right_index=True,
                       how='inner', suffixes=('_acc', '_veh'))

# Merge with Casualties
merged_data = pd.merge(merged_data, Casualties,
                       left_index=True, right_index=True,
                       how='inner', suffixes=('', '_cas'))

print("✅ Merging complete!")
print("Final merged shape:", merged_data.shape)

# Preview the merged dataset
display(merged_data.head(5))

# **3) Feture Selection**

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

df = merged_data.copy()

# Select numeric columns only
numeric_df = df.select_dtypes(include=['int64', 'float64'])

# Correlation with target (sorted by absolute value)
corr = numeric_df.corr()['Accident_Severity'].abs().sort_values(ascending=False)

print("Correlation (absolute) with Accident_Severity:\n")
print(corr)

# Select features by threshold
threshold = 0.005
selected_features = corr[corr > threshold].index.tolist()

# Remove target itself
if 'Accident_Severity' in selected_features:
    selected_features.remove('Accident_Severity')

print("\n Selected features based on |correlation| > 0.05:\n")
print(selected_features)

# Heatmap of selected features
plt.figure(figsize=(12, 8))
sns.heatmap(numeric_df[selected_features + ['Accident_Severity']].corr(),
            cmap='coolwarm', annot=False)
plt.title("Correlation Heatmap of Selected Features")
plt.show()

# Final  dataset
final_df = df[selected_features + ['Accident_Severity']]

print("\n✅ Final dataset shape:", final_df.shape)
display(final_df.head())

In [ ]:
# Extract Final 10 Pre-Accident Features

final_10_features = [
    'Speed_limit',
    'Urban_or_Rural_Area',
    'Light_Conditions',
    'Road_Type',
    '1st_Road_Class',
    'Local_Authority_(District)',
    'Age_of_Driver',
    'Age_Band_of_Driver',
    'Sex_of_Driver',
    'Journey_Purpose_of_Driver'
]

# Keep only features that exist in selected_features AND in df
final_10_features = [f for f in final_10_features if f in df.columns]

print("🔥 Final 10 Selected Features:")
print(final_10_features)

# Create final ML dataset
final_df = df[final_10_features + ['Accident_Severity']]

print("\n✅ Final dataset shape:", final_df.shape)
display(final_df.head())


# **4) Create Mapping Dictionaries**

In [ ]:
import pickle


# MAPPING DICTIONARIES

urban_rural_map = {
    1: "Urban",
    2: "Rural",
    3: "Unallocated"
}
urban_rural_rev = {v: k for k, v in urban_rural_map.items()}

light_map = {
    1: "Daylight",
    4: "Darkness - lights lit",
    5: "Darkness - lights unlit",
    6: "Darkness - no lighting",
    7: "Darkness - lighting unknown",
    -1: "Data missing or out of range"
}
light_rev = {v: k for k, v in light_map.items()}

road_type_map = {
    1: "Roundabout",
    2: "One way street",
    3: "Dual carriageway",
    6: "Single carriageway",
    7: "Slip road",
    9: "Unknown",
    12: "One way street/Slip road",
    -1: "Data missing or out of range"
}
road_type_rev = {v: k for k, v in road_type_map.items()}

road_class_map = {
    0: "Not at junction or within 20 metres",
    1: "Motorway",
    2: "A(M)",
    3: "A",
    4: "B",
    5: "C",
    6: "Unclassified"
}
road_class_rev = {v: k for k, v in road_class_map.items()}

age_band_map = {
    1: "0-5",
    2: "6-10",
    3: "11-15",
    4: "16-20",
    5: "21-25",
    6: "26-35",
    7: "36-45",
    8: "46-55",
    9: "56-65",
    10: "66-75",
    11: "Over 75",
    -1: "Unknown"
}
age_band_rev = {v: k for k, v in age_band_map.items()}

sex_map = {
    1: "Male",
    2: "Female",
    3: "Not known",
    -1: "Missing"
}
sex_rev = {v: k for k, v in sex_map.items()}

journey_map = {
    1: "Journey as part of work",
    2: "Commuting to/from work",
    3: "Taking pupil to/from school",
    4: "Pupil riding to/from school",
    5: "Other",
    6: "Not known",
    15: "Other/Not known (2005-10)",
    -1: "Missing"
}
journey_rev = {v: k for k, v in journey_map.items()}

severity_map = {
    1: "Fatal",
    2: "Serious",
    3: "Slight"
}
severity_rev = {v: k for k, v in severity_map.items()}


# SAVE ALL MAPPINGS TO ONE OBJECT
input_mappings = {
    "urban_rural_rev": urban_rural_rev,
    "light_rev": light_rev,
    "road_type_rev": road_type_rev,
    "road_class_rev": road_class_rev,
    "age_band_rev": age_band_rev,
    "sex_rev": sex_rev,
    "journey_rev": journey_rev,
    "severity_map": severity_map,
    "severity_rev": severity_rev
}

pickle.dump(input_mappings, open("input_mapper.pkl", "wb"))

print("✅ Input mappings saved as input_mapper.pkl")

# **5) Data Preprocessing**

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.impute import SimpleImputer

final_features = [
    'Speed_limit',
    'Urban_or_Rural_Area',
    'Light_Conditions',
    'Road_Type',
    '1st_Road_Class',
    'Local_Authority_(District)',
    'Age_of_Driver',
    'Age_Band_of_Driver',
    'Sex_of_Driver',
    'Journey_Purpose_of_Driver'
]

final_features = [f for f in final_features if f in merged_data.columns]

df = merged_data.copy()


# IMPUTE MISSING VALUES
imputer = SimpleImputer(strategy='median')
df[final_features] = imputer.fit_transform(df[final_features])


# **6) Train Random Forest Model**

In [ ]:
# 📌 OPTION A: 3-CLASS MODEL


df_3 = df[final_features + ['Accident_Severity']].copy()
df_3 = df_3.dropna()

X3 = df_3[final_features]
y3 = df_3['Accident_Severity'] - 1   # 1,2,3 → 0,1,2

X3_train, X3_test, y3_train, y3_test = train_test_split(
    X3, y3, test_size=0.2, stratify=y3
)

model_3 = RandomForestClassifier(
    n_estimators=250,
    max_depth=14,
    class_weight='balanced',
    n_jobs=-1
)

model_3.fit(X3_train, y3_train)
pred_3 = model_3.predict(X3_test)
acc_3 = accuracy_score(y3_test, pred_3)

print("\n==============================")
print("🔵 OPTION A — 3-Class Accuracy")
print("==============================")
print(f"Accuracy: {acc_3*100:.2f}%")
print(classification_report(y3_test+1, pred_3+1,
        target_names=['Fatal(1)','Serious(2)','Slight(3)']))


# 📌 OPTION B: FATAL vs NON-FATAL (2 class)

df_bin1 = df[final_features + ['Accident_Severity']].copy()
df_bin1['binary'] = (df_bin1['Accident_Severity'] == 1).astype(int)

Xb1 = df_bin1[final_features]
yb1 = df_bin1['binary']

Xb1_train, Xb1_test, yb1_train, yb1_test = train_test_split(
    Xb1, yb1, test_size=0.2, stratify=yb1
)

model_b1 = RandomForestClassifier(
    n_estimators=250,
    max_depth=12,
    class_weight='balanced',
    n_jobs=-1
)

model_b1.fit(Xb1_train, yb1_train)
pred_b1 = model_b1.predict(Xb1_test)
acc_b1 = accuracy_score(yb1_test, pred_b1)

print("\n==============================")
print("🔴 OPTION B — Fatal vs Non-Fatal")
print("==============================")
print(f"Accuracy: {acc_b1*100:.2f}%")
print(classification_report(yb1_test, pred_b1,
        target_names=['Non-Fatal','Fatal']))

# 📌 OPTION C: Slight vs Serious+Fatal (2 class)

df_bin2 = df[final_features + ['Accident_Severity']].copy()
df_bin2['binary2'] = df_bin2['Accident_Severity'].apply(lambda x: 0 if x==3 else 1)

Xb2 = df_bin2[final_features]
yb2 = df_bin2['binary2']

Xb2_train, Xb2_test, yb2_train, yb2_test = train_test_split(
    Xb2, yb2, test_size=0.2, stratify=yb2
)

model_b2 = RandomForestClassifier(
    n_estimators=250,
    max_depth=14,
    class_weight='balanced',
    n_jobs=-1
)

model_b2.fit(Xb2_train, yb2_train)
pred_b2 = model_b2.predict(Xb2_test)
acc_b2 = accuracy_score(yb2_test, pred_b2)

print("\n==============================")
print("🟢 OPTION C — Slight vs Serious+Fatal")
print("==============================")
print(f"Accuracy: {acc_b2*100:.2f}%")
print(classification_report(yb2_test, pred_b2,
        target_names=['Slight','Serious+Fatal']))


# **7) Load Saved model**

In [ ]:
import pickle
import numpy as np
import pandas as pd

# Load saved model, mappings, features, and imputer
try:
    model = pickle.load(open("model.pkl", "rb"))
    mappings = pickle.load(open("input_mapper.pkl", "rb"))
    selected_features = pickle.load(open("selected_features.pkl", "rb"))
    imputer = pickle.load(open("imputer.pkl", "rb"))
    print("✅ All models and mappings loaded successfully!")
except FileNotFoundError as e:
    print(f"⚠️ Warning: {e}")
    print("Please run the training cell first to generate model files.")

def predict_severity(user_input):
    """
    Predict accident severity using the trained model.
    
    user_input should contain all required features. For features not provided,
    default values will be used based on the training data.
    
    Example:
    user_input = {
        "Speed_limit": 30,
        "Urban_or_Rural_Area": "Urban",
        "Light_Conditions": "Daylight",
        "Road_Type": "Roundabout",
        "1st_Road_Class": "A",
        "Local_Authority_(District)": 12,
        "Age_of_Driver": 25,
        "Age_Band_of_Driver": "21-25",
        "Sex_of_Driver": "Male",
        "Journey_Purpose_of_Driver": "Commuting to/from work"
    }
    """
    
    # Create a dataframe with all selected features, initialized with NaN
    feature_dict = {}
    
    # Map user input to feature names (handle both formats)
    input_mapping = {
        "Speed_limit": "Speed_limit",
        "Urban_or_Rural_Area": "Urban_or_Rural_Area",
        "Light_Conditions": "Light_Conditions",
        "Road_Type": "Road_Type",
        "1st_Road_Class": "1st_Road_Class",
        "Local_Authority_(District)": "Local_Authority_(District)",
        "Age_of_Driver": "Age_of_Driver",
        "Age_Band_of_Driver": "Age_Band_of_Driver",
        "Sex_of_Driver": "Sex_of_Driver",
        "Journey_Purpose_of_Driver": "Journey_Purpose_of_Driver"
    }
    
    # Initialize all features with NaN
    for feat in selected_features:
        feature_dict[feat] = np.nan
    
    # Fill in provided features
    for key, value in user_input.items():
        if key in input_mapping:
            feat_name = input_mapping[key]
            if feat_name in selected_features:
                # Convert categorical to numeric if needed
                if key == "Urban_or_Rural_Area":
                    feature_dict[feat_name] = mappings["urban_rural_rev"].get(value, np.nan)
                elif key == "Light_Conditions":
                    feature_dict[feat_name] = mappings["light_rev"].get(value, np.nan)
                elif key == "Road_Type":
                    feature_dict[feat_name] = mappings["road_type_rev"].get(value, np.nan)
                elif key == "1st_Road_Class":
                    feature_dict[feat_name] = mappings["road_class_rev"].get(value, np.nan)
                elif key == "Age_Band_of_Driver":
                    feature_dict[feat_name] = mappings["age_band_rev"].get(value, np.nan)
                elif key == "Sex_of_Driver":
                    feature_dict[feat_name] = mappings["sex_rev"].get(value, np.nan)
                elif key == "Journey_Purpose_of_Driver":
                    feature_dict[feat_name] = mappings["journey_rev"].get(value, np.nan)
                else:
                    feature_dict[feat_name] = value
        elif key in selected_features:
            # Direct feature name match
            feature_dict[key] = value
    
    # Create dataframe with correct feature order
    X = pd.DataFrame([feature_dict])[selected_features]
    
    # Handle missing values using the saved imputer
    X = imputer.transform(X)
    
    # Make prediction
    prediction = model.predict(X)[0]
    probabilities = model.predict_proba(X)[0]
    max_probability = probabilities.max()
    
    # Map prediction to label
    severity_label = mappings["severity_map"].get(prediction, "Unknown")
    
    return severity_label, round(max_probability, 3)


✅ All models and mappings loaded successfully!


# **8) Testing**

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

user_nonfatal = {
    "Speed_limit": 30,                                
    "Urban_or_Rural_Area": "Urban",                    
    "Light_Conditions": "Daylight",                    
    "Road_Type": "Roundabout",                         
    "1st_Road_Class": "C",                             
    "Local_Authority_(District)": 12,                  
    "Age_of_Driver": 45,                             
    "Age_Band_of_Driver": "36 - 45",                   
    "Sex_of_Driver": "Female",                         
    "Journey_Purpose_of_Driver": "Commuting to/from work"
}

pred, prob = predict_severity(user_nonfatal)
print("Predicted Severity:", pred, "| Probability:", prob)

Predicted Severity: Fatal | Probability: 0.433
